# rl19_spec

> Phase 19 (`specs/multiagent/phase19/`): arm 4's measures from the evaluation's sequences of 6 boards (`rl19_suite.seqCells`): specialization (Part B's variance-ratio index: the variance across agents of their share of turns spent sharing over its binomial expectation, minus its shuffled-turns null) at 8, 16 and 32 agents and its slope over log2 of team size; the division of labour, reported apart (Gorelick et al. 2004's index: the mutual information between agents and tasks over the tasks' marginal entropy); role persistence (the correlation of each agent's share of contesting turns between consecutive boards); and recovery after the most active contesters are removed and return.

In [ ]:
#| default_exp rl19_spec

In [ ]:
#| hide
from nbdev.showdoc import *

## Specialization, persistence and recovery

In [ ]:
#| export
import json

import numpy as np

from explRL.rl19_suite import SEQ_SIZES

NAN = float('nan')


def _f(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return NAN


def _agents(rs, nLiar, positions, col='agentShares'):
    "Per original honest agent over the boards at `positions`: (turns, count of col)."
    T, X = {}, {}
    for r in rs:
        if int(_f(r['pos'])) in positions:
            for a, n, x in zip(json.loads(r['agents']), json.loads(r['agentTurns']), json.loads(r[col])):
                if a >= nLiar and n:
                    T[a], X[a] = T.get(a, 0) + n, X.get(a, 0) + x
    return T, X


def seqIndex(rs, nLiar, positions, nNull=200, seed=0):
    """Specialization, Part B's variance-ratio index across the boards of one sequence: the variance across honest agents of
    their share of turns spent sharing over its binomial expectation, minus the mean of the same with the turns shuffled
    across agents."""
    T, X = _agents(rs, nLiar, positions)
    if len(T) < 2:
        return NAN
    n, x = np.array([T[a] for a in sorted(T)]), np.array([X[a] for a in sorted(T)])

    def idx(k):
        p = k.sum() / n.sum()
        b = np.mean(p * (1 - p) / n)
        return float(np.var(k / n) / b) if b > 0 else NAN
    flags, rng, cut = np.repeat([1.0, 0.0], [int(x.sum()), int(n.sum() - x.sum())]), np.random.default_rng(seed), np.cumsum(n)[:-1]
    null = [idx(np.array([f.sum() for f in np.split(rng.permutation(flags), cut)])) for _ in range(nNull)]
    return idx(x) - float(np.nanmean(null))


TASKS = ('contest', 'share', 'submit', 'open', 'wait')   # share: shares that are not contests; wait: every other turn


def taskMix(rs, nLiar, positions):
    "{original honest agent: counts of TASKS} over the boards at `positions`."
    out = {}
    for r in rs:
        if int(_f(r['pos'])) in positions:
            cols = [json.loads(r[c]) for c in ('agents', 'agentTurns', 'agentShares', 'agentContests', 'agentSubmits', 'agentOpens')]
            for a, n, sh, c, sb, op in zip(*cols):
                if a >= nLiar and n:
                    out[a] = out.get(a, 0) + np.array([c, sh - c, sb, op, n - sh - sb - op], float)
    return out


def dolIndex(rs, nLiar, positions=range(6)):
    """The division of labour (Gorelick et al. 2004): the mutual information between agents and TASKS over the tasks'
    marginal entropy, I(agent; task) / H(task), from the agents' turns across the boards at `positions` (1 when an
    agent's identity fixes its task, 0 when every agent has the same mix)."""
    M = np.array(list(taskMix(rs, nLiar, positions).values()))
    if len(M) < 2 or M.sum() <= 0:
        return NAN
    p = M / M.sum()
    pa, pt = p.sum(1, keepdims=True), p.sum(0)
    nz = p > 0
    H = float(-(pt[pt > 0] * np.log(pt[pt > 0])).sum())
    return float((p[nz] * np.log(p[nz] / (pa * pt)[nz])).sum()) / H if H > 0 else NAN


def sequences(ix, m, test):
    "{sequence id: its rows}."
    out = {}
    for r in ix.get(f'{m}|{test}', []):
        out.setdefault(int(_f(r['seqId'])), []).append(r)
    return out


def _nl(rs):
    return int(_f(rs[0]['nLiar']))


def specialization(ix, m, N, nNull=200):
    "Mean over the intact sequences at N agents of seqIndex (Part B's variance ratio) over all 6 boards."
    v = [seqIndex(rs, _nl(rs), range(6), nNull) for rs in sequences(ix, m, f'seq{N}').values()]
    return float(np.nanmean(v)) if v else NAN


def division(ix, m, N):
    "Mean over the intact sequences at N agents of dolIndex (Gorelick) over all 6 boards."
    v = [dolIndex(rs, _nl(rs)) for rs in sequences(ix, m, f'seq{N}').values()]
    return float(np.nanmean(v)) if np.isfinite(v).any() else NAN


def persistence(ix, m):
    """Role persistence: the correlation across agents of each agent's share of contesting turns (contests / turns) between
    consecutive boards (1-2, ..., 5-6), averaged over board pairs, intact sequences and sizes (pairs without spread skipped)."""
    out = []
    for N in SEQ_SIZES:
        for rs in sequences(ix, m, f'seq{N}').values():
            for p in range(5):
                (t1, x1), (t2, x2) = _agents(rs, _nl(rs), (p,), 'agentContests'), _agents(rs, _nl(rs), (p + 1,), 'agentContests')
                ag = sorted(set(t1) & set(t2))
                a, b = [x1[i] / t1[i] for i in ag], [x2[i] / t2[i] for i in ag]
                if len(ag) > 2 and np.std(a) > 0 and np.std(b) > 0:
                    out.append(float(np.corrcoef(a, b)[0, 1]))
    return float(np.mean(out)) if out else NAN


def recovery(ix, m, nNull=200):
    """At 16 agents with the removal: the index on boards 4-5 (removed) and on board 6 (returned) minus boards 1-3's, and
    the take-up: the remaining agents' contests per turn on boards 4-5 minus on boards 1-3."""
    rem, ret, up = [], [], []
    for rs in sequences(ix, m, 'seqrm16').values():
        nl, base = _nl(rs), seqIndex(rs, _nl(rs), range(3), nNull)
        rem.append(seqIndex(rs, nl, (3, 4), nNull) - base)
        ret.append(seqIndex(rs, nl, (5,), nNull) - base)
        (t0, c0), (t1, c1) = _agents(rs, nl, range(3), 'agentContests'), _agents(rs, nl, (3, 4), 'agentContests')
        stay = sorted(set(t1) & set(t0))
        up.append(sum(c1[a] for a in stay) / max(1, sum(t1[a] for a in stay)) - sum(c0[a] for a in stay) / max(1, sum(t0[a] for a in stay)))
    m_ = lambda v: float(np.nanmean(v)) if v else NAN
    return dict(removed=m_(rem), returned=m_(ret), takeUp=m_(up))


def specMeasures(ix, m, nNull=200):
    "Arm 4's measures: specialization at 8, 16 and 32 agents and its slope over log2 size, the division of labour, persistence and recovery."
    sp = {N: specialization(ix, m, N, nNull) for N in SEQ_SIZES}
    x = [N for N in SEQ_SIZES if np.isfinite(sp[N])]
    slope = float(np.polyfit(np.log2(x), [sp[N] for N in x], 1)[0]) if len(x) > 1 else NAN
    return dict(spec=sp, specSlope=slope, dol={N: division(ix, m, N) for N in SEQ_SIZES}, persistence=persistence(ix, m),
                recovery=recovery(ix, m, nNull))

## Tests

In [ ]:
# specialization (Part B's variance ratio): fixed roles above mixed ones; the removal and the take-up
from explRL.rl19_spec import *
from explRL.rl19_measures import index
R = lambda m, t, **k: dict(level=f'{m}|{t}', **k)
def seqRows(m, shares, N=8, nl=2, turns=40, removed=(), test='seq8', contests=lambda a, p: int(a == 7), submits=None):
    out = []
    for p in range(6):
        ag = [i for i in range(N) if not (p in (3, 4) and i in removed)]
        out.append(R(m, test, seqId=900, pos=p, nLiar=nl, agents=json.dumps(ag), agentTurns=json.dumps([turns] * len(ag)),
                     agentShares=json.dumps([0 if a < nl else shares(a, p) for a in ag]), agentContests=json.dumps([contests(a, p) for a in ag]),
                     agentSubmits=json.dumps([0 if a < nl else (submits or (lambda a, p: (turns - shares(a, p)) // 2))(a, p) for a in ag]), agentOpens=json.dumps([0] * len(ag))))
    return out
fixed, mixed = seqRows('f', lambda a, p: 30 if a % 2 else 4), seqRows('x', lambda a, p: 17 if (a + p) % 2 else 16)
ixs = index(fixed + mixed)
assert seqIndex(fixed, 2, range(6), 50) > 5 > seqIndex(mixed, 2, range(6), 50) and specialization(ixs, 'f', 8, 50) > 5
assert np.isnan(specialization(ixs, 'nobody', 8))
# the division of labour (Gorelick): fixed roles above mixed ones, between 0 and 1; 1 when each agent does one task
assert dolIndex(mixed, 2) < 0.02 < 0.2 < dolIndex(fixed, 2) < 1 and division(ixs, 'f', 8) == dolIndex(fixed, 2)
one = seqRows('o', lambda a, p: 40 if a % 2 else 0, contests=lambda a, p: 0, submits=lambda a, p: 0 if a % 2 else 40)
assert abs(dolIndex(one, 2) - 1) < 1e-12 and taskMix(one, 2, range(6))[3].tolist() == [0, 240, 0, 0, 0]
# role persistence: each agent's share of contesting turns, correlated between consecutive boards
keep = seqRows('k', lambda a, p: 10, contests=lambda a, p: a)              # the same contesters on every board
flip = seqRows('s', lambda a, p: 10, contests=lambda a, p: a if p % 2 else 10 - a)
pers = persistence(index(keep + flip), 'k'), persistence(index(keep + flip), 's')
assert abs(pers[0] - 1) < 1e-12 and abs(pers[1] + 1) < 1e-12 and abs(persistence(ixs, 'f') - 1) < 1e-12
rec = recovery(index(seqRows('f', lambda a, p: 30 if a % 2 else 4, removed=(7,), test='seqrm16')), 'f', 50)
assert np.isfinite(rec['removed']) and rec['takeUp'] == 0.0           # agent 7 made every contest; the others take none up

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()